# 12. WaveNet GAN + Lambert W Gaussianization

Same **WaveNet-based GAN** architecture as NB07, but with the **QuantGAN-style 3-stage preprocessing**
pipeline from Wiese et al. (2020):

1. **StandardScaler** — center and normalize log returns
2. **Gaussianize** — Lambert W inverse transform to remove heavy tails (Goerg 2011, 2015)
3. **StandardScaler** — re-standardize the Gaussianized output

### Key difference from NB07
| Aspect | NB07 (WaveNet GAN) | NB12 (this notebook) |
|--------|-------------------|----------------------|
| Preprocessing | MinMaxScaler [0,1] | StandardScaler → Gaussianize → StandardScaler |
| Generator output | sigmoid (bounded) | **tanh** (approx ±3σ Gaussian range) |
| Data range | [0, 1] | ~N(0,1) (unbounded, standardized) |
| Tail handling | None | Lambert W removes excess kurtosis |

Everything else (WaveNet architecture, gated causal convolutions, moment loss,
adaptive LR scheduling) is identical to NB07 for a controlled comparison.

## 1. Setup & Imports

In [ ]:
# Import libraries
import os
import site
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Suppress TF C++ info/warning spam (must be set BEFORE importing TF)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# XLA fix for pip-bundled CUDA toolkit
_cuda_nvcc_dir = os.path.join(
    site.getsitepackages()[0] if site.getsitepackages() else
    os.path.join(os.path.dirname(os.__file__), '..', 'site-packages'),
    'nvidia', 'cuda_nvcc'
)
os.environ['XLA_FLAGS'] = f'--xla_gpu_cuda_data_dir={_cuda_nvcc_dir}'

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Multiply, Add, Concatenate, Dense,
    Dropout, GaussianNoise, LayerNormalization
)
from tensorflow.keras.models import Model, load_model

import sys
sys.path.append('..')
from utils.gaussianize import Gaussianize

# Set seeds
tf.random.set_seed(42)
np.random.seed(42)

# Suppress Python-level TF logs
tf.get_logger().setLevel('ERROR')

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

## 2. Data Loading & Lambert W Preprocessing

3-stage pipeline (from QuantGAN paper, Section 3):  
1. **StandardScaler** — center and normalize (needed for IGMM convergence)  
2. **Gaussianize** — Lambert W inverse transform to remove heavy tails  
3. **StandardScaler** — re-standardize the Gaussianized output  

Unlike NB07's MinMaxScaler [0,1], the output is ~N(0,1) — the generator
output activation must be unbounded (tanh, not sigmoid).

In [ ]:
# Load SP500 data (same source as NB07)
data = pd.read_csv('../data/raw/sp500.csv', index_col='Date', parse_dates=True)
data = data.apply(pd.to_numeric, errors='coerce')

close_prices = data['Close']
log_returns = np.log(close_prices / close_prices.shift(1)).dropna()
log_returns_array = log_returns.values.reshape(-1, 1)

print(f"Raw log returns: {len(log_returns_array)} observations")
print(f"  Range: [{log_returns_array.min():.4f}, {log_returns_array.max():.4f}]")
print(f"  Mean:  {log_returns_array.mean():.6f}")
print(f"  Std:   {log_returns_array.std():.6f}")
print(f"  Kurtosis: {float(pd.Series(log_returns_array.ravel()).kurtosis()):.2f}")

In [ ]:
# ── 3-stage preprocessing: StandardScaler → Gaussianize → StandardScaler ──

# Stage 1: standardize (needed for IGMM convergence)
scaler1 = StandardScaler()
stage1 = scaler1.fit_transform(log_returns_array)

# Stage 2: Gaussianize (Lambert W inverse — remove heavy tails)
gaussianizer = Gaussianize(max_iter=100, tol=1e-6)
stage2 = gaussianizer.fit_transform(stage1)

# Stage 3: re-standardize
scaler2 = StandardScaler()
log_returns_scaled = scaler2.fit_transform(stage2.reshape(-1, 1))

print(f"Gaussianize delta: {gaussianizer.delta_[0]:.6f}")
print(f"After preprocessing:")
print(f"  Range: [{log_returns_scaled.min():.4f}, {log_returns_scaled.max():.4f}]")
print(f"  Mean:  {log_returns_scaled.mean():.6f}, Std: {log_returns_scaled.std():.6f}")
print(f"  Kurtosis: {float(pd.Series(log_returns_scaled.ravel()).kurtosis()):.2f}")

# Verify round-trip
recovered = scaler1.inverse_transform(
    gaussianizer.inverse_transform(
        scaler2.inverse_transform(log_returns_scaled)
    ).reshape(-1, 1)
)
roundtrip_err = np.max(np.abs(recovered - log_returns_array))
print(f"  Round-trip max error: {roundtrip_err:.2e}")

In [ ]:
# Create sequences (same windowing as NB07)
sequence_length = 24  # Match LSTM TimeGAN for fair comparison
sequences = []
for i in range(len(log_returns_scaled) - sequence_length):
    sequences.append(log_returns_scaled[i:i+sequence_length])

sequences = np.array(sequences, dtype=np.float32)
print(f"Data shape: {sequences.shape}")
print(f"Scaled range: [{sequences.min():.4f}, {sequences.max():.4f}]")

## 3. WaveNet Building Blocks

Identical to NB07 — gated causal convolutions with skip connections.

In [ ]:
###############################################
# WaveNet Residual Block with Optional Gaussian Noise
# Adapted from: wavenet_gan_uncond_noise.py
###############################################
def wavenet_residual_block(input_tensor, nfilt, dilation_rate, 
                           residual_noise_std=None, seed=None):
    """Gated causal convolution block with skip connection.
    
    The core building block: uses tanh * sigmoid gating (like the 
    original WaveNet paper) instead of plain activations.
    """
    x = input_tensor
    
    # Project to nfilt channels if needed
    if x.shape[-1] != nfilt:
        x = Conv1D(filters=nfilt, kernel_size=1, padding='same')(x)
    
    # Optional noise injection for regularization
    if residual_noise_std:
        x = GaussianNoise(stddev=residual_noise_std, seed=seed)(x)
    
    # Gated activation: tanh(conv) * sigmoid(conv)
    tanh_out = Conv1D(filters=nfilt, kernel_size=3, dilation_rate=dilation_rate,
                      padding='causal', activation='tanh')(x)
    sigm_out = Conv1D(filters=nfilt, kernel_size=3, dilation_rate=dilation_rate,
                      padding='causal', activation='sigmoid')(x)
    gated = Multiply()([tanh_out, sigm_out])
    
    # Skip and residual outputs
    skip_out = Conv1D(filters=nfilt, kernel_size=1, padding='same')(gated)
    residual = Conv1D(filters=nfilt, kernel_size=1, padding='same')(gated)
    residual_out = Add()([x, residual])
    
    return residual_out, skip_out


###############################################
# WaveNet Block — stack of dilated residual blocks
###############################################
def wavenet_block(input_tensor, nfilt, dilation_rates=None,
                  residual_noise_std=None, seed=None):
    """Stack of dilated causal convolutions with exponentially increasing dilation."""
    if dilation_rates is None:
        dilation_rates = [1, 2, 4, 8, 16]
    
    skip_connections = []
    x = input_tensor
    for i, dilation in enumerate(dilation_rates):
        x, skip = wavenet_residual_block(
            x, nfilt, dilation,
            residual_noise_std=residual_noise_std,
            seed=(seed + i) if seed else None
        )
        skip_connections.append(skip)
    return Add()(skip_connections)


###############################################
# Deep WaveNet — multiple WaveNet blocks stacked
###############################################
def deep_wavenet(input_tensor, nfilt, n_stacks=2,
                 residual_noise_std=None, seed=None):
    """Stack multiple WaveNet blocks for deeper temporal modeling."""
    x = input_tensor
    for i in range(n_stacks):
        x = wavenet_block(
            x, nfilt,
            residual_noise_std=residual_noise_std,
            seed=(seed + 100 + i) if seed else None
        )
    return x


print("WaveNet building blocks defined.")

## 4. Build Generator and Discriminator

**Key change**: Generator output uses `tanh` (not `sigmoid`) since Gaussianized
data is ~N(0,1) with range approximately [-3, +3], not [0, 1].

In [ ]:
###############################################
# Generator — tanh output for Gaussianized data
###############################################
def build_wavenet_generator(sequence_length=24, feature_dim=1, latent_dim=32, nfilt=32,
                            n_stacks=2, latent_noise_std=0.01, 
                            residual_noise_std=0.01, seed=None):
    """WaveNet-based generator for unconditional financial time-series.
    
    Output activation: tanh (data is Gaussianized ~N(0,1), range ~[-3,3]).
    NB07 uses sigmoid for MinMaxScaled [0,1] data.
    """
    latent_in = Input(shape=(sequence_length, latent_dim), name='latent_input')
    
    x = latent_in
    
    # Latent augmentation (from swim repo generator)
    if latent_noise_std:
        x = GaussianNoise(latent_noise_std, name='latent_noise',
                          seed=seed)(x)
    
    # Process through WaveNet
    x = deep_wavenet(x, nfilt, n_stacks=n_stacks,
                     residual_noise_std=residual_noise_std, seed=seed)
    
    # ▶ tanh output — Gaussianized data is ~N(0,1), NOT [0,1]
    output = Dense(feature_dim, activation='tanh', name='generator_output')(x)
    
    model = Model(inputs=latent_in, outputs=output, name='WaveNet_Generator')
    return model


###############################################
# Discriminator (identical to NB07)
###############################################
def build_wavenet_discriminator(sequence_length=24, feature_dim=1, nfilt=32,
                                n_stacks=2, residual_noise_std=None,
                                dropout_rate=None, seed=None):
    """WaveNet-based discriminator for financial time-series."""
    data_in = Input(shape=(sequence_length, feature_dim), name='data_input')
    
    x = deep_wavenet(data_in, nfilt, n_stacks=n_stacks,
                     residual_noise_std=residual_noise_std, seed=seed)
    
    # Optional dropout
    if dropout_rate:
        x = Dropout(dropout_rate, seed=seed)(x)
    
    # Per-timestep real/fake output
    output = Dense(1, activation='sigmoid', name='real_fake_output')(x)
    
    model = Model(inputs=data_in, outputs=output, name='WaveNet_Discriminator')
    return model


print("Generator (tanh output) and Discriminator builders defined.")

## 5. Instantiate Models and Training Setup

In [ ]:
# Hyperparameters (identical to NB07)
LATENT_DIM = 32        # Noise vector dimension per timestep
NFILT = 32             # WaveNet filter count
N_STACKS = 2           # Number of stacked WaveNet blocks
FEATURE_DIM = 1        # Single feature (log return)
SEQUENCE_LENGTH = 24   # Match LSTM TimeGAN

LR_GENERATOR = 0.0002
LR_DISCRIMINATOR = 0.0001
BETA_1 = 0.5
BATCH_SIZE = 128
EPOCHS = 2000

# Build models
generator = build_wavenet_generator(
    sequence_length=SEQUENCE_LENGTH,
    feature_dim=FEATURE_DIM,
    latent_dim=LATENT_DIM,
    nfilt=NFILT,
    n_stacks=N_STACKS,
    latent_noise_std=0.01,
    residual_noise_std=0.01,
    seed=42
)

discriminator = build_wavenet_discriminator(
    sequence_length=SEQUENCE_LENGTH,
    feature_dim=FEATURE_DIM,
    nfilt=NFILT,
    n_stacks=N_STACKS,
    residual_noise_std=0.01,
    dropout_rate=0.1,
    seed=42
)

# Optimizers (same as NB07)
gen_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LR_GENERATOR, beta_1=BETA_1
)
disc_optimizer = tf.keras.optimizers.Adam(
    learning_rate=LR_DISCRIMINATOR, beta_1=BETA_1
)

generator.summary()
print()
discriminator.summary()

## 6. Training Loop

Same training procedure as NB07:  
- Adversarial + Huber reconstruction + moment matching loss  
- Gradient clipping to [-1, 1]  
- Balanced adaptive LR scheduling  
- TensorBoard logging

In [ ]:
import datetime
from utils.models_utils import (smooth_positive_labels, smooth_negative_labels,
                                BalancedAdaptiveLearningRateSchedule,
                                compute_moment_loss)

# Loss functions
bce = tf.keras.losses.BinaryCrossentropy()
huber = tf.keras.losses.Huber(delta=0.1)

# Lambda weights for generator loss components
LAMBDA_ADV = 1.0
LAMBDA_RECON = 100.0
LAMBDA_MOMENT = 1.0

# Balanced Adaptive LR (same config as NB07)
adaptive_lr = BalancedAdaptiveLearningRateSchedule(
    initial_gen_lr=LR_GENERATOR,
    initial_disc_lr=LR_DISCRIMINATOR,
    adjustment_factor=1.1,
    tolerance=0.4,
    min_lr=1e-5,
    max_lr=5e-4,
    max_lr_ratio=5.0,
)
LR_ADJUST_EVERY = 50

# TensorBoard
log_dir = os.path.join('..', 'logs', 'wavenet_gan_lambert',
                       datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))
tb_writer = tf.summary.create_file_writer(log_dir)
print(f"TensorBoard log dir: {log_dir}")
print(f"  → Launch: tensorboard --logdir {os.path.dirname(log_dir)}")


@tf.function
def train_step(real_data, batch_size):
    """Single training step — identical to NB07."""
    
    noise_gen = tf.random.normal(
        shape=(batch_size, SEQUENCE_LENGTH, LATENT_DIM), mean=0.0, stddev=1.0
    )
    noise_disc = tf.random.normal(
        shape=(batch_size, SEQUENCE_LENGTH, LATENT_DIM), mean=0.0, stddev=1.0
    )
    
    with tf.GradientTape() as disc_tape, tf.GradientTape() as gen_tape:
        fake_data_gen = generator(noise_gen, training=True)
        fake_data_disc = generator(noise_disc, training=True)
        
        real_output = discriminator(real_data, training=True)
        fake_output_disc = discriminator(fake_data_disc, training=True)
        fake_output_gen = discriminator(fake_data_gen, training=True)
        
        # Label smoothing
        real_labels = smooth_positive_labels(tf.ones_like(real_output))
        fake_labels = smooth_negative_labels(tf.zeros_like(fake_output_disc))
        
        # Discriminator loss
        d_loss_real = tf.reduce_mean(
            tf.keras.losses.binary_crossentropy(real_labels, real_output))
        d_loss_fake = tf.reduce_mean(
            tf.keras.losses.binary_crossentropy(fake_labels, fake_output_disc))
        total_disc_loss = d_loss_real + d_loss_fake
        
        # Generator losses
        gen_real_labels = smooth_positive_labels(tf.ones_like(fake_output_gen))
        g_loss_adv = tf.reduce_mean(
            tf.keras.losses.binary_crossentropy(gen_real_labels, fake_output_gen))
        
        g_loss_recon = tf.reduce_mean(
            tf.keras.losses.huber(real_data, fake_data_gen, delta=0.1))
        
        g_loss_moment = compute_moment_loss(real_data, fake_data_gen)
        
        total_gen_loss = (LAMBDA_ADV * g_loss_adv
                          + LAMBDA_RECON * g_loss_recon
                          + LAMBDA_MOMENT * g_loss_moment)
    
    disc_grads = disc_tape.gradient(total_disc_loss, discriminator.trainable_variables)
    gen_grads = gen_tape.gradient(total_gen_loss, generator.trainable_variables)
    
    disc_grads = [tf.clip_by_value(g, -1.0, 1.0) if g is not None else g 
                  for g in disc_grads]
    gen_grads = [tf.clip_by_value(g, -1.0, 1.0) if g is not None else g 
                 for g in gen_grads]
    
    disc_optimizer.apply_gradients(
        zip(disc_grads, discriminator.trainable_variables))
    gen_optimizer.apply_gradients(
        zip(gen_grads, generator.trainable_variables))
    
    return {
        'disc_loss': total_disc_loss,
        'gen_loss': total_gen_loss,
        'disc_real': d_loss_real,
        'disc_fake': d_loss_fake,
        'gen_adv': g_loss_adv,
        'gen_recon': g_loss_recon,
        'gen_moment': g_loss_moment,
    }


print("Training step compiled.")

In [ ]:
# Create tf.data.Dataset for mini-batch training
dataset = tf.data.Dataset.from_tensor_slices(sequences)
dataset = dataset.shuffle(buffer_size=len(sequences), seed=42)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

# Training loop
print(f"Training WaveNet GAN (Lambert W) for {EPOCHS} epochs...")
print(f"Dataset: {len(sequences)} sequences, {len(sequences)//BATCH_SIZE} steps/epoch")
print(f"Generator LR: {LR_GENERATOR}, Discriminator LR: {LR_DISCRIMINATOR}")
print(f"Adaptive LR re-balance every {LR_ADJUST_EVERY} epochs")
print(f"Lambda: adv={LAMBDA_ADV}, recon={LAMBDA_RECON}, moment={LAMBDA_MOMENT}")
print(f"Preprocessing: StandardScaler → Gaussianize(δ={gaussianizer.delta_[0]:.4f}) → StandardScaler")
print("="*70)

history = {
    'disc_loss': [], 'gen_loss': [],
    'disc_real': [], 'disc_fake': [],
    'gen_adv': [], 'gen_recon': [], 'gen_moment': [],
    'lr_gen': [], 'lr_disc': []
}

for epoch in range(EPOCHS):
    epoch_metrics = {key: [] for key in history if key not in ('lr_gen', 'lr_disc')}
    
    for batch in dataset:
        batch_size = tf.shape(batch)[0]
        metrics = train_step(batch, batch_size)
        
        for key in epoch_metrics:
            epoch_metrics[key].append(metrics[key].numpy())
    
    for key in epoch_metrics:
        history[key].append(np.mean(epoch_metrics[key]))

    # Balanced Adaptive LR
    if (epoch + 1) % LR_ADJUST_EVERY == 0:
        new_gen_lr, new_disc_lr = adaptive_lr(
            history['disc_loss'][-1] / 2.0,
            history['gen_adv'][-1]
        )
        gen_optimizer.learning_rate.assign(new_gen_lr)
        disc_optimizer.learning_rate.assign(new_disc_lr)

    history['lr_gen'].append(float(gen_optimizer.learning_rate))
    history['lr_disc'].append(float(disc_optimizer.learning_rate))

    # TensorBoard
    with tb_writer.as_default():
        tf.summary.scalar('loss/disc_total', history['disc_loss'][-1], step=epoch)
        tf.summary.scalar('loss/disc_real', history['disc_real'][-1], step=epoch)
        tf.summary.scalar('loss/disc_fake', history['disc_fake'][-1], step=epoch)
        tf.summary.scalar('loss/gen_total', history['gen_loss'][-1], step=epoch)
        tf.summary.scalar('loss/gen_adversarial', history['gen_adv'][-1], step=epoch)
        tf.summary.scalar('loss/gen_reconstruction', history['gen_recon'][-1], step=epoch)
        tf.summary.scalar('loss/gen_moment', history['gen_moment'][-1], step=epoch)
        tf.summary.scalar('lr/generator', history['lr_gen'][-1], step=epoch)
        tf.summary.scalar('lr/discriminator', history['lr_disc'][-1], step=epoch)

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}/{EPOCHS} | "
              f"D: {history['disc_loss'][-1]:.4f} "
              f"(r:{history['disc_real'][-1]:.3f} f:{history['disc_fake'][-1]:.3f}) | "
              f"G: {history['gen_loss'][-1]:.4f} "
              f"(adv:{history['gen_adv'][-1]:.3f} rec:{history['gen_recon'][-1]:.4f} "
              f"mom:{history['gen_moment'][-1]:.3f}) | "
              f"LR {history['lr_gen'][-1]:.1e}/{history['lr_disc'][-1]:.1e}")

tb_writer.flush()
print("\nTraining completed!")

## 7. Training Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total losses
axes[0].plot(history['disc_loss'], label='Discriminator', alpha=0.7)
axes[0].plot(history['gen_loss'], label='Generator', alpha=0.7)
axes[0].set_title('Total Losses')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Discriminator breakdown
axes[1].plot(history['disc_real'], label='D(real)', alpha=0.7)
axes[1].plot(history['disc_fake'], label='D(fake)', alpha=0.7)
axes[1].set_title('Discriminator Losses')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

# Generator breakdown
axes[2].plot(history['gen_adv'], label='Adversarial', alpha=0.7)
axes[2].plot(history['gen_recon'], label='Reconstruction', alpha=0.7)
axes[2].plot(history['gen_moment'], label='Moment', alpha=0.7)
axes[2].set_title('Generator Losses')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].legend()
axes[2].grid(True)

plt.suptitle('WaveNet GAN + Lambert W — Training Curves', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Generate Synthetic Data & Inverse Transform

In [ ]:
n_samples = 100

# Generate synthetic sequences in Gaussianized space
Z_synthetic = tf.random.normal(
    shape=(n_samples, SEQUENCE_LENGTH, LATENT_DIM), mean=0.0, stddev=1.0
)
synthetic_sequences = generator(Z_synthetic, training=False)
synthetic_gaussianized = synthetic_sequences.numpy()  # (n_samples, 24, 1)

print(f"Synthetic (Gaussianized space):")
print(f"  Range: [{synthetic_gaussianized.min():.4f}, {synthetic_gaussianized.max():.4f}]")
print(f"  Mean:  {synthetic_gaussianized.mean():.6f}")
print(f"  Std:   {synthetic_gaussianized.std():.6f}")

# ── Inverse preprocessing: StandardScaler⁻¹ → Gaussianize⁻¹ → StandardScaler⁻¹ ──
synth_flat = synthetic_gaussianized.reshape(-1, 1)

# Inverse stage 3 (StandardScaler)
inv3 = scaler2.inverse_transform(synth_flat)

# Inverse stage 2 (Gaussianize — re-add heavy tails)
inv2 = gaussianizer.inverse_transform(inv3.ravel()).reshape(-1, 1)

# Inverse stage 1 (StandardScaler → original log-return scale)
synthetic_rescaled = scaler1.inverse_transform(inv2).reshape(n_samples, SEQUENCE_LENGTH)

print(f"\nSynthetic (log-return space):")
print(f"  Range: [{synthetic_rescaled.min():.6f}, {synthetic_rescaled.max():.6f}]")
print(f"  Mean:  {synthetic_rescaled.mean():.6f}")
print(f"  Std:   {synthetic_rescaled.std():.6f}")
print(f"\nReal log returns:")
print(f"  Mean:  {log_returns.mean():.6f}")
print(f"  Std:   {log_returns.std():.6f}")

In [ ]:
# Plot synthetic vs real sequences
plot_idx = np.linspace(0, len(sequences) - 1, 5, dtype=int)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Synthetic
for i in range(5):
    axes[0].plot(synthetic_rescaled[i], alpha=0.7, label=f'Synthetic {i+1}')
axes[0].set_title('WaveNet GAN + Lambert — Synthetic Sequences')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Log Return')
axes[0].legend()
axes[0].grid(True)

# Real (inverse-transform from Gaussianized sequences)
real_gauss = sequences[plot_idx].reshape(-1, 1)
real_inv3 = scaler2.inverse_transform(real_gauss)
real_inv2 = gaussianizer.inverse_transform(real_inv3.ravel()).reshape(-1, 1)
real_plot = scaler1.inverse_transform(real_inv2).reshape(5, SEQUENCE_LENGTH)
for i in range(5):
    axes[1].plot(real_plot[i], alpha=0.7, label=f'Real {i+1}')
axes[1].set_title('Real Sequences')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Log Return')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Synthetic stats — mean: {synthetic_rescaled.mean():.6f}, "
      f"std: {synthetic_rescaled.std():.6f}, "
      f"min: {synthetic_rescaled.min():.6f}, max: {synthetic_rescaled.max():.6f}")
print(f"Real stats     — mean: {log_returns.mean():.6f}, "
      f"std: {log_returns.std():.6f}, "
      f"min: {log_returns.min():.6f}, max: {log_returns.max():.6f}")

## 9. Save Models & Data

In [ ]:
os.makedirs('../models', exist_ok=True)

generator.save('../models/wavenet_lambert_generator.keras')
discriminator.save('../models/wavenet_lambert_discriminator.keras')

generator.save_weights('../models/wavenet_lambert_generator.weights.h5')
discriminator.save_weights('../models/wavenet_lambert_discriminator.weights.h5')

# Save synthetic data in Gaussianized space
np.save('../data/processed/synthetic_wavenet_lambert.npy', synthetic_sequences.numpy())

# Save log-return-scale output for NB09 evaluation
# NB09 expects MinMaxScaled [0,1], so re-scale for compatibility
from sklearn.preprocessing import MinMaxScaler
minmax_for_eval = MinMaxScaler()
minmax_for_eval.fit(log_returns_array)
synth_minmax = minmax_for_eval.transform(
    synthetic_rescaled.reshape(-1, 1)
).reshape(n_samples, SEQUENCE_LENGTH, 1)
np.save('../data/processed/synthetic_wavenet_lambert_minmax.npy', synth_minmax)

# Also save raw log-return CSV
pd.DataFrame(
    synthetic_rescaled.flatten(), columns=['Synthetic_LogReturn']
).to_csv('../data/synthetic/wavenet_lambert_synthetic_full.csv', index=False)

print("Saved:")
print("  models/wavenet_lambert_generator.keras")
print("  models/wavenet_lambert_discriminator.keras")
print(f"  data/processed/synthetic_wavenet_lambert.npy — {synthetic_sequences.numpy().shape}")
print(f"  data/processed/synthetic_wavenet_lambert_minmax.npy — {synth_minmax.shape}")
print(f"  data/synthetic/wavenet_lambert_synthetic_full.csv")